# Environmental impact of our experiments

Every experiment runs inside `track()` (`include/tracking.py`), which logs its wall time and CPU time to
`results/runtime_log.json`. Energy is estimated as wall time $\times$ an assumed 45 W laptop draw under load, and
CO$_2$e with the GB grid average of 0.124 kg/kWh. Some notebooks ran at the same time, so the wall-time estimate counts
the shared periods more than once: treat the total as an upper bound. The CPU-time column gives a lower estimate that
splits the 45 W across the machine's logical cores.

In [ ]:
import os
import matplotlib.pyplot as plt

from include.tracking import load_json, CPU_POWER_W, GRID_INTENSITY
from include.figures import use_paper_style, savefig
from include.plots_sustainability import plot_energy

use_paper_style()
log = load_json("runtime_log.json")
cores = os.cpu_count()
print(f"{'experiment':<22} {'wall (min)':>10} {'CPU (min)':>10} {'Wh (wall)':>10} {'Wh (CPU)':>9} {'g CO2e':>7}")
for name, r in log.items():
    wh_cpu = r["cpu_s"] * CPU_POWER_W / cores / 3600
    print(f"{name:<22} {r['wall_s'] / 60:>10.1f} {r['cpu_s'] / 60:>10.1f} {r['energy_kWh'] * 1000:>10.2f} "
          f"{wh_cpu:>9.2f} {r['kgCO2e'] * 1000:>7.2f}")
wall_h = sum(r["wall_s"] for r in log.values()) / 3600
kwh = sum(r["energy_kWh"] for r in log.values())
kwh_cpu = sum(r["cpu_s"] for r in log.values()) * CPU_POWER_W / cores / 3.6e6
print(f"\ntotal: {wall_h:.1f} h of wall time, {kwh * 1000:.0f} Wh (upper bound) / {kwh_cpu * 1000:.0f} Wh (CPU time), "
      f"{kwh * GRID_INTENSITY * 1000:.0f} g CO2e (upper bound)")

In [ ]:
fig = plot_energy(log)
savefig(fig, "sustainability_energy")
plt.show()